In [1]:
from pathlib import Path
import pickle
import random

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

from config import RESULTS_DIR

DATA_DIR = Path(RESULTS_DIR) / "random_trajectories"
CHECKPOINT_DIR = Path(RESULTS_DIR) / "transformer"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / "hexapod_transformer_fitness_8D.pt"

SEED = 42
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 1e-4
TRAIN_SPLIT = 0.8
MAX_SEQ_LEN = 2000
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Using device: {DEVICE}")
print(f"Loading trajectories from: {DATA_DIR}")


Using device: cuda
Loading trajectories from: /home/joze/Mestrado/Dissertacao/Pybullet/results/random_trajectories


In [2]:
class RandomTrajectoryFitnessDataset(Dataset):
    """Lazy dataset that keeps only file paths in memory and loads sequences on demand.

    Assumes one trajectory per file (dict or list-with-one-entry). This avoids loading
    tens of thousands of pickles into RAM at once which caused the kernel OOM/crash.
    """

    def __init__(self, file_paths):
        self.file_paths = list(file_paths)
        if not self.file_paths:
            raise RuntimeError('No trajectory files provided')

        # Determine length and sample a single file to infer state_dim
        self._infer_state_dim()

    def _load_entry_from_file(self, path):
        try:
            with open(path, "rb") as f:
                loaded = pickle.load(f)
        except (EOFError, pickle.UnpicklingError):
            print(f"Skipping corrupted file: {path}")
            return None

        entries = loaded if isinstance(loaded, list) else [loaded]

        for entry in entries:

            # New nested format
            if "result" in entry:
                entry = entry["result"]

            if (
                "detailed_log" in entry
                and entry.get("fitness") is not None
            ):
                return entry

        return None

    def _infer_state_dim(self):
        for p in self.file_paths:
            entry = self._load_entry_from_file(p)
            if entry is None:
                continue
            detailed_log = entry.get('detailed_log', [])
            if len(detailed_log) == 0:
                continue
            step = detailed_log[0]
            position = np.asarray(step.get('position', []), dtype=np.float32)
            orientation = np.asarray(step.get('orientation', []), dtype=np.float32)
            contacts = np.asarray(step.get('contacts', []), dtype=np.float32)
            torques = np.asarray(step.get('torques', []), dtype=np.float32)
            self.state_dim = int(position.size + orientation.size + contacts.size + torques.size)
            return
        # Fallback
        self.state_dim = 0

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        path = self.file_paths[idx]
        entry = self._load_entry_from_file(path)
        if entry is None:
            # Return an empty sequence with zero fitness (shouldn't happen for valid dataset)
            return np.zeros((1, self.state_dim), dtype=np.float32), np.float32(0.0)

        detailed_log = entry.get('detailed_log', [])
        fitness = float(entry.get('fitness', 0.0))

        seq = []
        for step in detailed_log:
            position = np.asarray(step.get('position', []), dtype=np.float32)
            orientation = np.asarray(step.get('orientation', []), dtype=np.float32)
            contacts = np.asarray(step.get('contacts', []), dtype=np.float32)
            torques = np.asarray(step.get('torques', []), dtype=np.float32)
            state = np.concatenate([position, orientation, contacts, torques]).astype(np.float32)
            seq.append(state)

        if not seq:
            return np.zeros((1, self.state_dim), dtype=np.float32), np.float32(fitness)

        seq = np.stack(seq)
        return seq, np.float32(fitness)


def load_random_trajectory_files(data_dir: Path):
    return sorted(data_dir.glob('traj_*.pkl'))


def compute_feature_stats_sampled(file_paths, sample_count=500):
    """Compute mean/std by randomly sampling up to `sample_count` files and accumulating state statistics.

    This avoids loading the entire corpus into memory.
    """
    file_paths = list(file_paths)
    n_files = len(file_paths)
    if n_files == 0:
        raise RuntimeError('No files to compute stats from')

    sample_count = min(sample_count, n_files)
    sampled = random.sample(file_paths, sample_count)

    sum_ = None
    sumsq = None
    total = 0

    for p in sampled:
        with open(p, 'rb') as f:
            loaded = pickle.load(f)
        entries = loaded if isinstance(loaded, list) else [loaded]
        entry = None
        entry = None

        for e in entries:

            # New nested format
            if "result" in e:
                e = e["result"]

            if "detailed_log" in e and e.get("fitness") is not None:
                entry = e
                break

        if entry is None:
            continue
        seq = []
        for step in entry['detailed_log']:
            position = np.asarray(step.get('position', []), dtype=np.float64)
            orientation = np.asarray(step.get('orientation', []), dtype=np.float64)
            contacts = np.asarray(step.get('contacts', []), dtype=np.float64)
            torques = np.asarray(step.get('torques', []), dtype=np.float64)
            state = np.concatenate([position, orientation, contacts, torques]).astype(np.float64)
            seq.append(state)
        if not seq:
            continue
        seq = np.stack(seq, axis=0)  # (T, D)
        if sum_ is None:
            D = seq.shape[1]
            sum_ = np.zeros(D, dtype=np.float64)
            sumsq = np.zeros(D, dtype=np.float64)
        sum_ += seq.sum(axis=0)
        sumsq += (seq ** 2).sum(axis=0)
        total += seq.shape[0]

    if total == 0:
        raise RuntimeError('No timesteps found while sampling files for stats')

    mean = sum_ / total
    var = (sumsq / total) - (mean ** 2)
    std = np.sqrt(np.maximum(var, 1e-12))
    std[std < 1e-6] = 1.0
    return mean.astype(np.float32), std.astype(np.float32)


# collate_fn factory remains the same and will be used below


In [3]:
class HexapodTransformer(nn.Module):

    def __init__(
        self,
        state_dim,
        d_model=64,
        nhead=4,
        num_layers=4,
        dim_feedforward=256,
        dropout=0.1,
        max_seq_len=2000,
        bd_dim=8,
    ):

        super().__init__()

        self.input_proj = nn.Linear(
            state_dim,
            d_model
        )

        self.cls_token = nn.Parameter(
            torch.randn(1, 1, d_model)
        )

        self.pos_embedding = nn.Parameter(
            torch.randn(1, max_seq_len + 1, d_model)
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers,
        )

        self.shared_norm = nn.LayerNorm(d_model)

        # ==================================================
        # 8D BEHAVIOR DESCRIPTOR HEAD
        # ==================================================

        self.bd_head = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Linear(64, bd_dim),
        )

        # ==================================================
        # FITNESS HEAD
        # ==================================================

        self.fitness_head = nn.Sequential(
            nn.Linear(bd_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, states, padding_mask=None):

        x = self.input_proj(states)

        batch_size, seq_len, _ = x.shape

        cls_tokens = self.cls_token.expand(
            batch_size,
            -1,
            -1
        )

        x = torch.cat([cls_tokens, x], dim=1)

        if seq_len + 1 > self.pos_embedding.shape[1]:
            raise ValueError(
                f'Sequence length {seq_len} exceeds '
                f'max_seq_len {self.pos_embedding.shape[1] - 1}'
            )

        x = x + self.pos_embedding[:, :seq_len + 1]

        if padding_mask is not None:
            cls_mask = torch.zeros( (padding_mask.shape[0], 1), dtype=torch.bool, device=padding_mask.device)
            padding_mask = torch.cat( [cls_mask, padding_mask], dim=1)

        x = self.transformer(x, src_key_padding_mask=padding_mask)

        cls_output = x[:, 0]
        cls_output = self.shared_norm(cls_output)

        bd = self.bd_head(cls_output)

        fitness_pred = self.fitness_head(bd).squeeze(-1)
        return fitness_pred, bd

In [4]:
trajectory_files = load_random_trajectory_files(DATA_DIR)
if not trajectory_files:
    raise FileNotFoundError(f'No pickle files found in {DATA_DIR}')

print(f'Found {len(trajectory_files)} trajectory files')

# lazy dataset keeps only file paths
dataset = RandomTrajectoryFitnessDataset(trajectory_files)
print(f'Loaded {len(dataset)} trajectories')
print(f'State dimension: {dataset.state_dim}')

train_size = max(1, int(len(dataset) * TRAIN_SPLIT))
val_size = len(dataset) - train_size
if val_size == 0:
    val_size = 1
    train_size = len(dataset) - val_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED),
)

# Compute feature stats by sampling files from the training subset (dataset is lazy)
train_indices = getattr(train_dataset, 'indices', None)
if train_indices is None:
    # Fallback: use first `train_size` files
    train_file_paths = trajectory_files[:train_size]
else:
    train_file_paths = [dataset.file_paths[i] for i in train_indices]

feature_mean, feature_std = compute_feature_stats_sampled(train_file_paths, sample_count=500)

# Inline collate_fn using sampled mean/std (avoid dependency on previous cell state)
feature_mean_t = torch.tensor(feature_mean, dtype=torch.float32)
feature_std_t = torch.tensor(feature_std, dtype=torch.float32)

def collate_fn(batch):
    sequences, fitness = zip(*batch)
    lengths = torch.tensor([seq.shape[0] for seq in sequences], dtype=torch.long)
    max_len = int(lengths.max().item())
    feature_dim = sequences[0].shape[-1]

    states = torch.zeros((len(sequences), max_len, feature_dim), dtype=torch.float32)
    padding_mask = torch.ones((len(sequences), max_len), dtype=torch.bool)

    for i, seq in enumerate(sequences):
        seq_tensor = torch.tensor(seq, dtype=torch.float32)
        seq_tensor = (seq_tensor - feature_mean_t) / feature_std_t
        seq_len = seq_tensor.shape[0]
        states[i, :seq_len] = seq_tensor
        padding_mask[i, :seq_len] = False

    fitness = torch.tensor(fitness, dtype=torch.float32)
    return states, padding_mask, lengths, fitness

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f'Train samples: {len(train_dataset)}')
print(f'Validation samples: {len(val_dataset)}')
print(f'Checkpoint path: {CHECKPOINT_PATH}')


Found 11239 trajectory files
Loaded 11239 trajectories
State dimension: 31
Train samples: 8991
Validation samples: 2248
Checkpoint path: /home/joze/Mestrado/Dissertacao/Pybullet/results/transformer/hexapod_transformer_fitness_8D.pt


In [5]:
model = HexapodTransformer(
    state_dim=dataset.state_dim,
    d_model=64,
    nhead=4,
    num_layers=4,
    dim_feedforward=256,
    dropout=0.1,
    max_seq_len=MAX_SEQ_LEN,
    bd_dim=8,
).to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE
)

loss_fn = nn.MSELoss()

# ==========================================================
# LOSS WEIGHTS
# ==========================================================

DIVERSITY_WEIGHT = 0.01
L2_BD_WEIGHT = 0.001

# ==========================================================
# EARLY STOPPING
# ==========================================================

patience = 5
epochs_without_improvement = 0

best_val_loss = float('inf')
best_epoch = -1

history = []

for epoch in range(1, EPOCHS + 1):

    # ======================================================
    # TRAIN
    # ======================================================

    model.train()

    train_loss = 0.0
    train_fitness_loss = 0.0
    train_div_loss = 0.0

    for states, padding_mask, lengths, fitness_target in train_loader:

        states = states.to(DEVICE)
        padding_mask = padding_mask.to(DEVICE)
        fitness_target = fitness_target.to(DEVICE)

        # --------------------------------------------------
        # FORWARD
        # --------------------------------------------------

        pred, bd = model(
            states,
            padding_mask=padding_mask
        )

        # --------------------------------------------------
        # FITNESS LOSS
        # --------------------------------------------------

        fitness_loss = loss_fn(
            pred,
            fitness_target
        )

        # --------------------------------------------------
        # DIVERSITY LOSS
        # --------------------------------------------------

        bd_std = torch.std(
            bd,
            dim=0
        ).mean()

        diversity_loss = -bd_std

        # --------------------------------------------------
        # L2 BD LOSS
        # --------------------------------------------------

        l2_bd_loss = torch.mean(
            bd ** 2
        )

        # --------------------------------------------------
        # TOTAL LOSS
        # --------------------------------------------------

        loss = (
            fitness_loss
            + DIVERSITY_WEIGHT * diversity_loss
            + L2_BD_WEIGHT * l2_bd_loss
        )

        # --------------------------------------------------
        # BACKPROP
        # --------------------------------------------------

        optimizer.zero_grad()

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        # --------------------------------------------------
        # LOGGING
        # --------------------------------------------------

        batch_size = states.size(0)

        train_loss += loss.item() * batch_size
        train_fitness_loss += fitness_loss.item() * batch_size
        train_div_loss += diversity_loss.item() * batch_size

    # ======================================================
    # VALIDATION
    # ======================================================

    model.eval()

    val_loss = 0.0
    val_fitness_loss = 0.0
    val_div_loss = 0.0

    with torch.no_grad():

        for states, padding_mask, lengths, fitness_target in val_loader:

            states = states.to(DEVICE)
            padding_mask = padding_mask.to(DEVICE)
            fitness_target = fitness_target.to(DEVICE)

            pred, bd = model(
                states,
                padding_mask=padding_mask
            )

            fitness_loss = loss_fn(
                pred,
                fitness_target
            )

            bd_std = torch.std(
                bd,
                dim=0
            ).mean()

            diversity_loss = -bd_std

            l2_bd_loss = torch.mean(
                bd ** 2
            )

            loss = (
                fitness_loss
                + DIVERSITY_WEIGHT * diversity_loss
                + L2_BD_WEIGHT * l2_bd_loss
            )

            batch_size = states.size(0)

            val_loss += loss.item() * batch_size
            val_fitness_loss += fitness_loss.item() * batch_size
            val_div_loss += diversity_loss.item() * batch_size

    # ======================================================
    # NORMALIZE
    # ======================================================

    train_loss /= len(train_dataset)
    train_fitness_loss /= len(train_dataset)
    train_div_loss /= len(train_dataset)

    val_loss /= len(val_dataset)
    val_fitness_loss /= len(val_dataset)
    val_div_loss /= len(val_dataset)

    # ======================================================
    # HISTORY
    # ======================================================

    history.append({
        'epoch': epoch,

        'train_loss': train_loss,
        'train_fitness_loss': train_fitness_loss,
        'train_diversity_loss': train_div_loss,

        'val_loss': val_loss,
        'val_fitness_loss': val_fitness_loss,
        'val_diversity_loss': val_div_loss,
    })

    # ======================================================
    # PRINT
    # ======================================================

    print(
        f'Epoch {epoch:03d} | '
        f'train={train_loss:.6f} | '
        f'val={val_loss:.6f} | '
        f'fit={val_fitness_loss:.6f} | '
        f'div={val_div_loss:.6f}'
    )

    # ======================================================
    # SAVE BEST + EARLY STOPPING
    # ======================================================

    if val_loss < best_val_loss:

        # New best
        best_val_loss = val_loss
        best_epoch = epoch

        # Reset early stopping counter
        epochs_without_improvement = 0

        torch.save(
            {
                'model_state_dict':
                    model.state_dict(),

                'state_dim':
                    dataset.state_dim,

                'feature_mean':
                    feature_mean,

                'feature_std':
                    feature_std,

                'best_val_loss':
                    best_val_loss,

                'best_epoch':
                    best_epoch,

                'history':
                    history,

                'config': {
                    'd_model': 64,
                    'nhead': 4,
                    'num_layers': 4,
                    'dim_feedforward': 256,
                    'dropout': 0.1,
                    'max_seq_len': MAX_SEQ_LEN,
                    'bd_dim': 8,
                },
            },
            CHECKPOINT_PATH,
        )

        print('  -> New best model saved.')

    else:

        # No improvement
        epochs_without_improvement += 1

        print(
            f'  -> No improvement '
            f'({epochs_without_improvement}/{patience})'
        )

    # ======================================================
    # EARLY STOPPING
    # ======================================================

    if epochs_without_improvement >= patience:

        print(
            f'\nEarly stopping at epoch {epoch}. '
            f'Validation loss did not improve for '
            f'{patience} consecutive epochs.'
        )

        break


# ==========================================================
# DONE
# ==========================================================

print(
    f'Best validation loss: '
    f'{best_val_loss:.6f} '
    f'at epoch {best_epoch}'
)

/home/joze/Mestrado/Dissertacao/Pybullet/bulletenv/lib/python3.11/site-packages/torch/nn/modules/transformer.py:505: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Epoch 001 | train=0.016950 | val=0.006649 | fit=0.010805 | div=-0.449981
  -> New best model saved.
Epoch 002 | train=0.004611 | val=0.000487 | fit=0.008648 | div=-0.924255
  -> New best model saved.
Epoch 003 | train=-0.003264 | val=-0.007636 | fit=0.005765 | div=-1.652315
  -> New best model saved.
Epoch 004 | train=-0.010381 | val=-0.013811 | fit=0.005152 | div=-2.705635
  -> New best model saved.
Epoch 005 | train=-0.015341 | val=-0.018153 | fit=0.003962 | div=-3.637075
  -> New best model saved.
Epoch 006 | train=-0.018255 | val=-0.019437 | fit=0.003818 | div=-4.173586
  -> New best model saved.
Epoch 007 | train=-0.019702 | val=-0.020316 | fit=0.003537 | div=-4.557269
  -> New best model saved.
Epoch 008 | train=-0.020444 | val=-0.021447 | fit=0.002822 | div=-4.729560
  -> New best model saved.
Epoch 009 | train=-0.021300 | val=-0.022029 | fit=0.002386 | div=-4.772286
  -> New best model saved.
Epoch 010 | train=-0.021456 | val=-0.021657 | fit=0.002318 | div=-4.907626
  -> No imp

In [6]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
print(f"Saved transformer checkpoint to: {CHECKPOINT_PATH}")
print(f"Checkpoint keys: {sorted(checkpoint.keys())}")

sample_states, sample_mask, sample_lengths, sample_target = next(iter(val_loader))
model.eval()
with torch.no_grad():
    sample_pred, sample_latent = model(sample_states.to(DEVICE), padding_mask=sample_mask.to(DEVICE))

print(f"Example predictions: {sample_pred[:5].detach().cpu().numpy()}")
print(f"Example targets: {sample_target[:5].numpy()}")
print(f"Latent shape: {tuple(sample_latent.shape)}")


Saved transformer checkpoint to: /home/joze/Mestrado/Dissertacao/Pybullet/results/transformer/hexapod_transformer_fitness_8D.pt
Checkpoint keys: ['best_epoch', 'best_val_loss', 'config', 'feature_mean', 'feature_std', 'history', 'model_state_dict', 'state_dim']
Example predictions: [0.17832384 0.24459478 0.01400076 0.01578215 0.38742563]
Example targets: [0.1666886  0.22750638 0.03149763 0.02024144 0.39462778]
Latent shape: (32, 8)
